# 08.01 ONNX、OM 与 ATC

本分册用一个小模型走通 `PyTorch → ONNX → OM → 推理` 的过程，观察每个文件格式和工具的作用。


## 实验原理

训练态的 `.pth` 依赖 PyTorch 和原始模型代码；昇腾部署使用 OM。两种模型在计算图、运行时和输入约束上不同。

| | 训练态（`.pth`） | 部署态（`.om`） |
| --- | --- | --- |
| 计算图 | 动态图，随 Python 逐行构建 | 静态图，编译期确定 |
| 保存内容 | 权重 + 结构代码 + 优化器状态 | 权重 + 已编排的执行序列 |
| 运行依赖 | PyTorch 与原始模型代码 | AscendCL runtime |
| 输入 shape | 可变 | 编译期固定，或声明取值范围 |
| 图优化 | 基本不做 | 算子融合、内存复用、权重重排 |

本分册经 ONNX 导出。ONNX 文件同时保存计算图和权重，可供工具读取、检查和转换。


ONNX 是模型交换格式，ATC 将它编译成目标芯片可加载的 OM。

```text
   model.pth  ──torch.onnx.export──►  model.onnx  ──atc──►  model.om
       │                                   │                    │
   torch 前向                        onnxruntime 前向        msame 推理
       ▼                                   ▼                    ▼
     输出 A ────── 对比 ──────► 输出 B ────── 对比 ──────► 输出 C
```

| 环节 | 角色 | 产物 |
| --- | --- | --- |
| 训练框架模型 | 提供模型结构和权重 | `.pth` |
| ONNX | 脱离框架描述计算图 | `.onnx` |
| ATC | 面向昇腾芯片编译模型 | 转换日志与 OM |
| OM | 昇腾加载执行的离线模型 | `.om` |

OM 与目标芯片型号绑定。`--soc_version` 应填写本机实测值，目标型号变化后需要重新转换。


## 实验流程

### 1. 定义并检查训练框架模型

ONNX 导出和 ATC 转换是两步操作。前者使用 CPU 版 PyTorch 即可完成；ATC 转换需要昇腾开发套件。

先安装 ONNX 相关 Python 库。


In [ ]:
!pip install onnx onnxruntime onnxscript -i https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
import torch, onnx, onnxruntime, numpy as np

print("torch       ", torch.__version__)
print("onnx        ", onnx.__version__)
print("onnxruntime ", onnxruntime.__version__)
print("providers   ", onnxruntime.get_available_providers())

下面定义演示模型。BatchNorm 和 Dropout 用于观察 train 与 eval 状态的差别。


In [ ]:
import torch.nn as nn

class DemoNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(8)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(8 * 4 * 4, num_classes)

    def forward(self, x):
        x = self.relu(self.bn(self.conv(x)))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

model = DemoNet()
INPUT_SHAPE = (1, 3, 32, 32)
print(model(torch.randn(*INPUT_SHAPE)).shape)

`__init__` 中声明层，`forward` 定义数据流。ONNX 导出记录实际执行的算子序列；未在 `forward` 中调用的层不会进入 ONNX 图。

#### 保存并读取 `.pth`


In [ ]:
import os

os.makedirs("l08_workspace", exist_ok=True)
torch.save(model.state_dict(), "l08_workspace/demo_model.pth")

sd = torch.load("l08_workspace/demo_model.pth", map_location="cpu")
for k, v in sd.items():
    print("%-24s %-16s %s" % (k, tuple(v.shape), v.dtype))

列出的内容是权重张量和 BatchNorm 统计量，没有层连接方式。因此，读取 `.pth` 前需要先定义 `DemoNet`。

ONNX 将结构和权重写入同一文件，便于跨框架读取。

#### 固定推理状态


In [ ]:
torch.manual_seed(0)
x = torch.randn(*INPUT_SHAPE)

model.train()
with torch.no_grad():
    a1, a2 = model(x), model(x)

model.eval()
with torch.no_grad():
    b1, b2 = model(x), model(x)

print("train 模式两次前向的最大差异: %.6f" % (a1 - a2).abs().max())
print("eval  模式两次前向的最大差异: %.6f" % (b1 - b2).abs().max())
print("两种模式之间的最大差异     : %.6f" % (a1 - b1).abs().max())

train 模式下，Dropout 每次随机置零的位置不同，BatchNorm 也会更新 running stats。eval 模式关闭 Dropout，并使用固定统计量，便于重复对照。

导出前应显式调用 `model.eval()`。它使后续 PyTorch 对照推理与导出时的模型状态一致；`torch.onnx.export` 的 `training` 默认行为可通过本机 API 签名核验。


### 2. 导出 ONNX

ONNX（Open Neural Network Exchange）包含网络结构、权重参数、输入输出信息和算子信息。

#### 导出方式

PyTorch 是动态图，ONNX 是静态图。`torch.onnx.export` 会以示例输入执行一次前向，并记录实际走过的算子，因此需要 `dummy_input`。示例输入的 shape 和 dtype 要与模型相符。

如果模型中的控制流依赖输入数值，trace 只会记录本次走到的分支；循环次数也会被固定。需要保留控制流时，可考虑 `torch.jit.script`。

#### 最小可用导出


In [ ]:
model.eval()
dummy_input = torch.randn(*INPUT_SHAPE)

torch.onnx.export(model, dummy_input, "l08_workspace/tmp.onnx")

m = onnx.load("l08_workspace/tmp.onnx")
print("输入名:", [i.name for i in m.graph.input])
print("输出名:", [o.name for o in m.graph.output])

不指定可选参数也能导出，但输入和输出名称会自动生成。下游的 ATC 与 msame 通过名称定位张量，因此应显式指定 `input_names` 和 `output_names`。

#### 参数说明

| 参数 | 作用 | 说明 |
| --- | --- | --- |
| `model` | 待导出模型 | 应先调用 `eval()` |
| `args` | 示例输入 | shape 与 dtype 正确；多输入传 tuple |
| `f` | 输出路径 | |
| `input_names` / `output_names` | 节点命名 | 下游工具据此定位张量 |
| `opset_version` | 算子集版本 | 需在 CANN 支持范围内 |
| `dynamic_axes` | 声明可变维度 | 不写则 shape 固定 |
| `do_constant_folding` | 常量折叠 | 默认 `True` |
| `export_params` | 是否带权重 | 默认 `True` |

更改张量名称后，后续 ATC 命令中的输入名也要同步更新。


In [ ]:
torch.onnx.export(
    model,
    dummy_input,
    "l08_workspace/demo_model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    do_constant_folding=True,
)

print("%.1f KB" % (os.path.getsize("l08_workspace/demo_model.onnx") / 1024))

#### 动态维度

不写 `dynamic_axes` 时，所有维度固定为 `dummy_input` 的 shape。需要可变维度时可这样声明：

```python
dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
dynamic_axes={"input": [0], "output": [0]}
```

| 场景 | ONNX 导出 | ATC | msame |
| --- | --- | --- | --- |
| 静态 | 不设 `dynamic_axes` | `--input_shape` | 无需额外参数 |
| 动态 batch | `{0: "batch"}` | `--dynamic_batch_size` | `--dymBatch` |
| 动态分辨率 | `{2: "h", 3: "w"}` | `--dynamic_image_size` | 相应指定 |
| 完全动态 | 多维声明 | `--input_shape_range` | `--dymShape` + `--outputSize` |

本分册使用静态 shape。动态 shape 的 ATC 参数和推理参数在本 Lab 的后续分册展开。

#### 导出时的注意事项

下列写法会影响导出结果：

- `tensor.item()`、`tensor.data` 会将张量降为常量；
- dict 或 str 作输入会被当作常量，输入输出应使用 tensor；
- 嵌套的 list/tuple 会被展开；
- inplace 操作可能使导出结果与 PyTorch 不一致；
- `nn.Upsample` 等算子兼容性较差，可提高 `opset_version` 或改用等价写法。


### 3. 检查 ONNX 模型

检查模型格式、图结构和数值输出。


In [ ]:
m = onnx.load("l08_workspace/demo_model.onnx")
onnx.checker.check_model(m)

print("IR version   :", m.ir_version)
print("opset        :", [(op.domain or "ai.onnx", op.version) for op in m.opset_import])
print("producer     :", m.producer_name, m.producer_version)

`check_model` 只检查 ONNX 格式是否合法，不计算模型输出。


In [ ]:
from collections import Counter

def describe(vi):
    t = vi.type.tensor_type
    dims = [d.dim_param if d.dim_param else d.dim_value for d in t.shape.dim]
    return "%-8s %-8s %s" % (vi.name, onnx.TensorProto.DataType.Name(t.elem_type), dims)

for i in m.graph.input:
    print("输入", describe(i))
for o in m.graph.output:
    print("输出", describe(o))

print("\n算子:", dict(Counter(n.op_type for n in m.graph.node)))

确认输入名称与导出时一致；shape 中出现字符串（如 `batch`）表示动态维；算子清单可用于初步核对 CANN 支持情况。

需要查看图结构时，可使用 Netron。


In [ ]:
import onnxruntime as ort

sess_options = ort.SessionOptions()


sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1

sess = ort.InferenceSession(
    "l08_workspace/demo_model.onnx",
    sess_options=sess_options,
    providers=["CPUExecutionProvider"]
)

np.random.seed(0)
x_np = np.random.randn(*INPUT_SHAPE).astype(np.float32)

ort_out = sess.run(
    None,
    {"input": x_np}
)[0]

print("输出 shape:", ort_out.shape)
print(np.round(ort_out[0][:5], 6))

输入 dtype 使用 `np.float32`，与此处模型的 torch float32 对应。输入字典的 key 是导出时的 `input_names`；读取已有模型时，可用 `sess.get_inputs()[0].name` 获取。

`run` 的第一个参数指定输出，传 `None` 表示按顺序返回全部输出。

#### 对比 PyTorch 与 onnxruntime


In [ ]:
model.eval()
with torch.no_grad():
    torch_out = model(torch.from_numpy(x_np)).numpy()

print("最大绝对误差 %.3e" % np.abs(torch_out - ort_out).max())
np.testing.assert_allclose(torch_out, ort_out, rtol=1e-3, atol=1e-5)
print("对齐 ① 通过")

浮点运算的实现和累加顺序不同，输出通常不要求逐元素完全相等。`rtol=1e-3, atol=1e-5` 是此处可用的容差。

若两侧输出差异较大，先检查模型状态、输入 dtype 和 inplace 操作。


### 4. 使用 ATC 转换 OM

ATC（Ascend Tensor Compiler）将 ONNX 等模型编译为昇腾可执行的 OM 离线模型。转换过程包括模型解析、shape 推导、算子融合、内存规划和图优化。

以下命令需要昇腾开发套件。

#### 配置环境变量


运行下方命令确认 ATC 已在当前 shell 中可用：


In [ ]:
!source /usr/local/Ascend/ascend-toolkit/set_env.sh && atc --help | head -20

每个 `%%bash` 或 `!` cell 都会启动独立子进程。因此，每个调用 `atc` 的 cell 都需要重新执行 `source set_env.sh`。

开发环境与部署环境架构不同时，例如 x86 开发、aarch64 部署，开发环境需要安装对应架构的 toolkit。安装细节以 CANN 官方文档为准。

#### 确认 `soc_version`


In [ ]:
!npu-smi info

除 `npu-smi info` 外，也可用 ACL 接口读取芯片型号。

#### 执行转换

将下面的 `--soc_version` 改为上一步获得的实测值。


In [ ]:
import acl
acl.init()
print(acl.get_soc_name())

执行转换：


In [ ]:
%%bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_PROCESS_NUM=1

atc --model=l08_workspace/demo_model.onnx \
    --framework=5 \
    --output=l08_workspace/demo_model \
    --input_format=NCHW \
    --input_shape="input:1,3,32,32" \
    --soc_version=Ascend910B4 \
    --log=info

转换成功后会生成 `.om` 文件及元信息 JSON。报错时优先查看日志中的 `[ERROR]` 行。

静态 shape 场景下，`--input_shape` 格式为 `input_name:N,C,H,W`；多个输入用分号分隔：

```bash
--input_shape="input1:1,3,224,224;input2:1,100"
```

#### 常用 ATC 参数

| 参数 | 说明 |
| --- | --- |
| `--model` | 输入模型路径 |
| `--framework` | 框架类型；5=ONNX |
| `--output` | 输出路径（不含扩展名）；`.om` 自动补齐 |
| `--soc_version` | 目标芯片型号，应与运行环境一致 |
| `--input_shape` | 静态场景下指定输入 shape |
| `--dynamic_batch_size` | 支持的 batch 档位 |
| `--dynamic_image_size` | 支持的分辨率 |
| `--input_shape_range` | 完全动态范围 |
| `--log=level` | 日志级别：debug/info/warning/error |
| `--precision_mode` | 精度模式 |
| `--fusion_switch_file` | 算子融合开关配置 |


### 5. 使用 msame 推理

#### 获取并编译 msame

msame 源码位于 <https://gitee.com/ascend/tools/tree/master/msame>。

OM 保存优化后的计算图、重排后的权重和设备执行调度信息。它是二进制格式，运行时通过 AscendCL 加载。


克隆后编译：


In [ ]:
x_np.tofile("l08_workspace/input.bin")
print("字节数", os.path.getsize("l08_workspace/input.bin"))
print("核算  ", np.prod(INPUT_SHAPE), "元素 x 4 字节 =", np.prod(INPUT_SHAPE) * 4)

编译产物通常位于 `out/msame`。

#### 准备输入文件

msame 的输入是原始 `.bin` 文件，一个文件对应一个输入。静态 shape 时，OM 元信息已包含输入 shape。


In [ ]:

!/opt/atomgit/tools/msame/out/msame --model l08_workspace/demo_model.om \
      --input l08_workspace/input.bin \
      --output l08_workspace/msame_out \
      --outfmt BIN

`--input` 可以指定单个 bin 或目录；`--output` 下会按时间戳创建子目录。调试时可使用 `TXT` 输出格式，数值比较时使用 `BIN`。

#### 对比 onnxruntime 与 msame


In [ ]:
import glob

out_bin = sorted(glob.glob("l08_workspace/msame_out/*/output_0.bin"))[-1]
msame_out = np.fromfile(out_bin, dtype=np.float32).reshape(1, 10)

print("最大绝对误差 %.3e" % np.abs(ort_out - msame_out).max())
print("argmax 一致?", (ort_out.argmax() == msame_out.argmax()))

cos_sim = (ort_out * msame_out).sum() / (np.linalg.norm(ort_out) * np.linalg.norm(msame_out))
print("余弦相似度 %.6f" % cos_sim)

np.testing.assert_allclose(ort_out, msame_out, rtol=1e-2, atol=1e-3)
print("对齐 ② 通过")

默认精度模式可能使用 FP16，逐层累计误差会大于 PyTorch 与 onnxruntime 的误差。比较输出时，可依次查看分类结果的 argmax、余弦相似度和逐元素误差。

统计耗时时，将首次推理与稳定运行分开。首次包含模型加载、内存分配和算子初始化。


成功后，输出位于 `./out/<timestamp>_output_0.bin`；多输出模型依次使用 `_1` 等后缀。遇到错误时，可结合 ACL error code 和 CANN 文档定位。

#### msame 常用参数

| 参数 | 说明 |
| --- | --- |
| `--model` | OM 模型路径 |
| `--input` | 输入文件或目录；bin 格式 |
| `--output` | 输出目录 |
| `--device` | 设备 ID，默认 0 |
| `--dymBatch` / `--dymHW` / `--dymShape` | 动态场景下指定实际 shape |
| `--outputSize` | 动态场景下预分配输出 buffer |
| `--loop` | 重复推理次数 |
| `--debug` | 打印更多日志 |


## 实验扩展

1. 把演示模型改成动态 batch 重新导出，用本分册的图结构检查代码观察输入 shape 的变化。
2. 去掉 `model.eval()` 重新导出，比较 PyTorch 与 onnxruntime 的输出，并解释现象。
3. 把 `--input_shape` 中的输入名改为不存在的名称，记录 ATC 的报错。
4. 对比 ONNX 与 OM 的文件大小，结合精度模式分析差异。
5. 如果模型中的 `if` 分支依赖输入数值，trace 导出会发生什么？可以怎样处理？


## 参考答案


In [ ]:
!cat answer/L08-01_answer.txt